# ForestCast
## Forty-year climate fingerprints for forest carbon

**The climate trajectory matters, not just its average.**

ForestCast turns forty years of monthly environmental drivers into an interpretable feature fingerprint: long-term levels, variability, linear trends, four decade blocks, end-state forcing and seasonal structure. It pairs that representation with site-grouped evaluation and the correct final-state output units.

This notebook contributes a memory-efficient reader for the provided forcing archives, feature visualizations on real public inputs, and an optional labelled-training baseline. The input plots describe known covariates; they are not validation scores or predictions of hidden outcomes.

[Source and companion files](https://github.com/GISWLH/AImodel-EarthScience/tree/main/kaggle-geoscience/forestcast) · Author: **giswlh / Longhao Wang**, with AI assistance.


In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'figure.dpi':115,'axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold'})
OUTPUT=Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()/'outputs'
OUTPUT.mkdir(parents=True,exist_ok=True)


## From monthly drivers to a forest-growth fingerprint

For each input driver, the reference block contains nine summaries: annual mean and standard deviation, a 40-year slope, the final five-year mean, four decade means and bounded early-to-late relative change. A seasonal block adds the across-month standard deviation, minimum, maximum and the late-minus-early change.

For the 136 provided drivers this produces 1224 reference and 544 seasonal features. The full recipe also includes the seven initial ecosystem variables, age and state-per-age terms. The optional baseline below uses the fixed reference block; seasonal features are exposed for controlled extensions.

The competition asks for year-40 height and above-ground biomass. Keep all fifteen initial-age cohorts of a site in the same fold. When formatting outputs, divide height by 10.90645 and AGB by 3.39567.


In [ ]:
"""Interpretable climate-history fingerprints for forest ecosystem emulation.

A transparent reference pipeline for site-grouped research.
Only real historical/labelled training data are accepted. No automatic submissions.
"""
from pathlib import Path
import numpy as np
import pandas as pd

CARBON_AGES=[1,10,20,30,50,70,100,150,200,250,300,350,400,450,500]
CARBON_SCALE=np.array([10.90645,3.39567],dtype=np.float32)

def carbon_forcing_features(x):
    """Actual (site,40years,12months,features) -> reference + seasonal summaries."""
    assert x.ndim==4 and x.shape[1:3]==(40,12) and np.isfinite(x).all()
    annual=x.mean(axis=2,dtype=np.float32);centered_year=np.arange(40,dtype=np.float32)-19.5
    early=annual[:,:5].mean(1);late=annual[:,-5:].mean(1)
    reference=np.concatenate([annual.mean(1),annual.std(1),(annual*centered_year[None,:,None]).sum(1)/(centered_year@centered_year),late,*[annual[:,i:i+10].mean(1) for i in range(0,40,10)],np.clip((late-early)/(np.abs(early)+1e-6),-10,10)],axis=1)
    seasonal=x.mean(1)
    return reference.astype(np.float32),np.concatenate([seasonal.std(1),seasonal.min(1),seasonal.max(1),late-early],axis=1).astype(np.float32)

def carbon_site_split(site_ids,seed=42,validation_fraction=.2):
    ids=np.asarray(site_ids);assert len(np.unique(ids))==len(ids) and len(ids)>=10
    rng=np.random.default_rng(seed);order=rng.permutation(ids);n=max(2,int(len(ids)*validation_fraction))
    train,valid=np.sort(order[n:]),np.sort(order[:n]);assert not np.intersect1d(train,valid).size
    return train,valid

def carbon_training_core(data_root,max_sites=512,seed=42,rounds=200):
    """Fixed LightGBM final-year baseline; all15ages of a site have one split.

    For bounded educational execution use the first max_sites training rows.
    This subset is not representative global validation and must not be used as
    evidence of an improved competitive model. No test inputs are opened.
    """
    from lightgbm import LGBMRegressor
    root=Path(data_root)/'data_global';x=np.load(root/'glob_X_fea.npy',mmap_mode='r')
    n=min(max_sites,len(x));assert n>=10
    reference,seasonal=carbon_forcing_features(np.asarray(x[:n]))
    initial=[];target=[]
    for age in CARBON_AGES:
        y=np.load(root/f'glob_Y{age:03d}.npy',mmap_mode='r');assert y.shape[1]==41
        initial.append(np.asarray(y[:n,0,-1,:],dtype=np.float32));target.append(np.asarray(y[:n,40,-1,:2],dtype=np.float32))
    initial=np.stack(initial,1);target=np.stack(target,1);assert initial.shape==(n,15,7)
    ages=np.asarray(CARBON_AGES,dtype=np.float32)[None,:,None]
    rows=np.concatenate([np.repeat(reference,15,axis=0),initial.reshape(-1,7),(initial/ages).reshape(-1,7),np.tile(CARBON_AGES,n)[:,None]],axis=1).astype(np.float32)
    train,valid=carbon_site_split(np.arange(n),seed)
    train_rows=np.concatenate([np.arange(i*15,(i+1)*15) for i in train]);valid_rows=np.concatenate([np.arange(i*15,(i+1)*15) for i in valid])
    truth=target.reshape(-1,2)/CARBON_SCALE;prediction=np.empty((len(valid_rows),2))
    for k in range(2):
        model=LGBMRegressor(n_estimators=rounds,num_leaves=31,learning_rate=.05,min_child_samples=30,reg_alpha=.5,reg_lambda=1.,random_state=seed,n_jobs=2,verbosity=-1)
        model.fit(rows[train_rows],truth[train_rows,k]);prediction[:,k]=np.maximum(model.predict(rows[valid_rows]),0)
    assert np.isfinite(prediction).all()
    return {'kind':'bounded_real_training_core_execution','representative_validation':False,'official_score':False,'test_inputs_read':False,'training_sites':len(train),'held_sites':len(valid),'all_ages_grouped':True,'seed':seed,'rounds':rounds,'target_year':40,'output_order':['height','agb'],'official_unit_divisors':CARBON_SCALE.tolist(),'normalized_held_MSE':float(np.mean((prediction-truth[valid_rows])**2))}


In [ ]:
"""Bounded reads of public climate inputs; no future targets or fitted predictions."""
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

def read_npz_rows(path,key,rows=64):
    # Reading a prefix avoids materializing a multi-GB array just for an input plot.
    with zipfile.ZipFile(path) as z:
        with z.open(key+'.npy') as member:
            version=np.lib.format.read_magic(member)
            if version==(1,0):shape,fortran,dtype=np.lib.format.read_array_header_1_0(member)
            elif version==(2,0):shape,fortran,dtype=np.lib.format.read_array_header_2_0(member)
            else:raise ValueError('Unsupported NumPy header version')
            assert not fortran and not dtype.hasobject
            n=min(rows,shape[0]);needed=int(n*np.prod(shape[1:])*dtype.itemsize)
            raw=member.read(needed);assert len(raw)==needed
            return np.frombuffer(raw,dtype=dtype).reshape((n,*shape[1:])).copy(),shape

def inspect_public_forcing(root,rows=64):
    paths={s:list(Path(root).rglob('test_'+s+'.npz')) for s in ['ssp126','ssp585']}
    values={};shapes={};records=[]
    for scenario,files in paths.items():
        assert len(files)==1,'Attach CarbonGlobe public inputs or specify CARBON_PUBLIC_ROOT'
        forcing,shape=read_npz_rows(files[0],'test_x',rows);assert forcing.shape[1:3]==(40,12)
        initial,initial_shape=read_npz_rows(files[0],'test_y0',rows)
        assert initial.shape[1:]==(15,7) and np.isfinite(forcing).all()
        reference,seasonal=carbon_forcing_features(forcing)
        values[scenario]={'forcing':forcing,'initial':initial,'reference':reference,'seasonal':seasonal};shapes[scenario]=list(shape)
        records.append({'scenario':scenario,'inspected_sites':len(forcing),'provided_sites':shape[0],'years':shape[1],'months':shape[2],'drivers':shape[3],'reference_features':reference.shape[1],'seasonal_features':seasonal.shape[1]})
    return values,pd.DataFrame(records)


## Explore actual provided forcing without loading the whole archive

The following reader streams only the first 64 site rows from each NumPy archive member. It does not read future targets or fit a prediction model. This bounded input view is useful for checking feature shapes, temporal behavior and scenario shifts; it is not a global population estimate. Driver indices are kept explicit because this attachment does not provide a complete feature-name schema.


In [ ]:
PUBLIC_ROOT=os.environ.get('CARBON_PUBLIC_ROOT','/kaggle/input')
inputs,overview=inspect_public_forcing(PUBLIC_ROOT,rows=64)
display(overview)


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,3.6))
for channel,ax in enumerate(axes):
    for scenario,color in [('ssp126','#2274a5'),('ssp585','#ef8354')]:
        annual=inputs[scenario]['forcing'].mean(axis=(0,2))[:,channel]
        ax.plot(np.arange(1,41),annual,color=color,label=scenario)
    ax.set(title=f'Provided driver {channel}',xlabel='Year',ylabel='Provided input units');ax.grid(alpha=.2)
axes[-1].legend();fig.suptitle('Scenario trajectories in the inspected public inputs',y=1.04);plt.tight_layout();plt.show()


## Initial age profiles are part of the problem

The same climate history is evaluated for different initial forest ages. These are the supplied year-0 states, not the hidden year-40 outcomes. Plotting the known profiles helps catch axis-order and cohort mistakes before any model is trained.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,3.6))
for scenario,color in [('ssp126','#2274a5'),('ssp585','#ef8354')]:
    state=inputs[scenario]['initial'].mean(axis=0)
    for k,(ax,name) in enumerate(zip(axes,['Initial height','Initial AGB'])):
        ax.plot(CARBON_AGES,state[:,k],'-o',color=color,label=scenario);ax.set(xlabel='Initial age',ylabel='Provided state units',title=name)
axes[-1].legend();plt.tight_layout();plt.show()


## Optional: train the fixed site-grouped baseline

The competition attachment supplies future forcing and initial states. To fit the model, attach the labelled CarbonGlobe training files separately and set `CARBON_TRAIN_ROOT`. The expected layout is `data_global/glob_X_fea.npy` plus `glob_Y001.npy`, `glob_Y010.npy`, etc. The code uses the year40targets and assigns all ages of a site to one split.

The small default site cap makes this an inspectable training exercise. Expand the data coverage and predefine independent site/domain tests before treating a validation number as representative. No result is manufactured when labelled training data are absent.


In [ ]:
RUN_TRAINING=False
TRAIN_ROOT=os.environ.get('CARBON_TRAIN_ROOT','')
if RUN_TRAINING:
    if not TRAIN_ROOT:raise RuntimeError('Attach labelled training data and set CARBON_TRAIN_ROOT.')
    training_report=carbon_training_core(TRAIN_ROOT,max_sites=512,seed=42,rounds=200)
    display(pd.DataFrame([training_report]));(OUTPUT/'forestcast_training.json').write_text(json.dumps(training_report,indent=2))
else:
    print('Feature exploration complete. Labelled training is opt-in; no hidden-target prediction was produced.')


## Extensions worth testing

Compare the reference and seasonal blocks with the same site split; examine low-CO2 or unusual initial-state domains separately. A residual model can learn change from the initial state, but the initial and final representations must use consistent units. Avoid fitting a scaler or selecting a checkpoint on held-site observations.

**Credits:** the public [CarbonGlobe baseline by Amirhossein Karimiee](https://www.kaggle.com/code/amirhosseinkarimiee/notebook04039c2d24) informed the forcing-summary recipe. The competition's public README defines the input axes, age cohorts and output scaling. This code shares the feature/training method, not private weights or hidden outcomes. Dataset rights remain with their providers.
